# Feature Engineering

## Objective

Create predictive features for machine learning based stock selection.

## Features

### Momentum

- 21 Day Momentum
- 63 Day Momentum
- 126 Day Momentum

### Volatility

- 21 Day Volatility
- 63 Day Volatility

### Trend

- EMA Distance
- EMA Slope

### Mean Reversion

- RSI 14

### Market Features

- Market Momentum
- Market Volatility

### Target

Future 21 Day Return

In [1]:
import pandas as pd
import numpy as np

prices = pd.read_csv(
    "../nifty500_statergy/data/nifty500_close_clean.csv",
    index_col=0,
    parse_dates=True
)

In [2]:
returns = prices.pct_change()


mom21 = prices.pct_change(21)

mom63 = prices.pct_change(63)

mom126 = prices.pct_change(126)


vol21 = (
    returns
    .rolling(21)
    .std()
)

vol63 = (
    returns
    .rolling(63)
    .std()
)


ema50 = (
    prices
    .ewm(
        span=50,
        adjust=False
    )
    .mean()
)

ema_distance = (
    prices
    /
    ema50
    - 1
)

ema_slope = (
    ema50
    .pct_change(10)
)



def calculate_rsi(
    prices,
    period=14
):

    delta = prices.diff()

    gain = (
        delta
        .clip(lower=0)
    )

    loss = (
        -delta
        .clip(upper=0)
    )

    avg_gain = (
        gain
        .rolling(period)
        .mean()
    )

    avg_loss = (
        loss
        .rolling(period)
        .mean()
    )

    rs = (
        avg_gain
        /
        avg_loss
    )

    return (
        100
        -
        (
            100
            /
            (1 + rs)
        )
    )


rsi14 = calculate_rsi(
    prices
)

In [3]:
market_ret = (
    returns
    .mean(axis=1)
)

market_mom21 = (
    market_ret
    .rolling(21)
    .sum()
)

market_vol21 = (
    market_ret
    .rolling(21)
    .std()
)

In [4]:
future_return = (
    prices
    .shift(-21)
    /
    prices
    - 1
)

In [5]:
feature_df = (
    mom21
    .stack()
    .rename(
        "mom21"
    )
    .to_frame()
)

feature_df["mom63"] = (
    mom63.stack()
)

feature_df["mom126"] = (
    mom126.stack()
)

feature_df["vol21"] = (
    vol21.stack()
)

feature_df["vol63"] = (
    vol63.stack()
)

feature_df["ema_distance"] = (
    ema_distance.stack()
)

feature_df["ema_slope"] = (
    ema_slope.stack()
)

feature_df["rsi14"] = (
    rsi14.stack()
)

feature_df["future_return"] = (
    future_return.stack()
)

feature_df["market_mom21"] = (
    feature_df.index
    .get_level_values(0)
    .map(
        market_mom21
    )
)

feature_df["market_vol21"] = (
    feature_df.index
    .get_level_values(0)
    .map(
        market_vol21
    )
)

In [6]:
feature_df = (
    feature_df
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .dropna()
)

feature_df.head()

mom21     mom63    mom126     vol21     vol63  \
Date                                                                        
2015-07-07 3MINDIA.NS   -0.028814 -0.044600  0.264628  0.012119  0.025000   
           ABB.NS        0.105837  0.039956  0.075473  0.017977  0.017446   
           ACC.NS        0.049351 -0.091014  0.081320  0.013131  0.018228   
           AIAENG.NS     0.083120 -0.164473 -0.066578  0.015916  0.028475   
           APLAPOLLO.NS  0.023391 -0.116183  0.176847  0.025337  0.021545   

                         ema_distance  ema_slope      rsi14  future_return  \
Date                                                                         
2015-07-07 3MINDIA.NS       -0.000386   0.003061  51.521348       0.211150   
           ABB.NS            0.047123   0.010039  67.181723      -0.015551   
           ACC.NS            0.022314  -0.003859  72.808272      -0.071802   
           AIAENG.NS        -0.024456  -0.015916  63.326813      -0.036672   
           APLAPOLLO.NS     -0.018824  -0.017880  41.069316       0.120344   

                         market_mom21  market_vol21  
Date                                                 
2015-07-07 3MINDIA.NS        0.075832      0.008273  
           ABB.NS            0.075832      0.008273  
           ACC.NS            0.075832      0.008273  
           AIAENG.NS         0.075832      0.008273  
           APLAPOLLO.NS      0.075832      0.008273

In [7]:
print(
    feature_df.shape
)

feature_df.describe()

(845236, 11)


,mom21,mom63,mom126,vol21,vol63,ema_distance,ema_slope,rsi14,future_return,market_mom21,market_vol21
count,845236.000000,845236.000000,845236.000000,845236.000000,845236.000000,845236.000000,845236.000000,845236.000000,845236.000000,845236.000000,845236.000000
mean,0.021369,0.065840,0.137437,0.022409,0.023222,0.016000,0.007247,51.675368,0.020728,0.020605,0.010279
std,0.123794,0.258176,0.399483,0.010956,0.009295,0.090479,0.035275,17.337298,0.123313,0.063878,0.004740
min,-0.949254,-0.953425,-0.953425,0.000020,0.001411,-0.948600,-0.308245,0.000000,-0.800697,-0.469775,0.003270
25%,-0.047056,-0.070993,-0.088717,0.015091,0.016802,-0.032535,-0.012160,39.466500,-0.047473,-0.012671,0.007042
50%,0.012051,0.036470,0.074748,0.020001,0.021278,0.012418,0.005328,51.882470,0.011335,0.028596,0.009252
75%,0.078991,0.166067,0.277778,0.026819,0.027648,0.061328,0.024564,64.235379,0.078125,0.061518,0.012374
max,2.326034,24.676400,36.879419,0.253804,0.148890,1.148434,0.518894,100.000000,2.326034,0.223325,0.044555


In [12]:
feature_df = feature_df.clip(
    lower=feature_df.quantile(0.01),
    upper=feature_df.quantile(0.99),
    axis=1
)



In [13]:
feature_df.to_csv(
    "feature_df_clean.csv"
)

import os

print(
    os.path.getsize(
        "feature_df_clean.csv"
    ) / 1024 / 1024,
    "MB"
)

199.33883380889893 MB


In [15]:
feature_df = pd.read_csv(
    "feature_df_clean.csv",
    index_col=[0,1],
    parse_dates=[0]
)

feature_df.head()

mom21     mom63    mom126     vol21     vol63  \
Date                                                                        
2015-07-07 3MINDIA.NS   -0.028814 -0.044600  0.264628  0.012119  0.025000   
           ABB.NS        0.105837  0.039956  0.075473  0.017977  0.017446   
           ACC.NS        0.049351 -0.091014  0.081320  0.013131  0.018228   
           AIAENG.NS     0.083120 -0.164473 -0.066578  0.015916  0.028475   
           APLAPOLLO.NS  0.023391 -0.116183  0.176847  0.025337  0.021545   

                         ema_distance  ema_slope      rsi14  future_return  \
Date                                                                         
2015-07-07 3MINDIA.NS       -0.000386   0.003061  51.521348       0.211150   
           ABB.NS            0.047123   0.010039  67.181723      -0.015551   
           ACC.NS            0.022314  -0.003859  72.808272      -0.071802   
           AIAENG.NS        -0.024456  -0.015916  63.326813      -0.036672   
           APLAPOLLO.NS     -0.018824  -0.017880  41.069316       0.120344   

                         market_mom21  market_vol21  
Date                                                 
2015-07-07 3MINDIA.NS        0.075832      0.008273  
           ABB.NS            0.075832      0.008273  
           ACC.NS            0.075832      0.008273  
           AIAENG.NS         0.075832      0.008273  
           APLAPOLLO.NS      0.075832      0.008273